# Ancestral reconstruction sharpness versus tree depth

How concentrated are marginal ancestral character distributions from root to tips? Compare available FT2 alphabets, amino acids and 3Di, using each strategy's **own alignment, fitted model and tree**. Sharpness is confidence, not accuracy against known ancestors.

Run in the `foldtree2` environment. This notebook only reads completed results; it never launches inference or modifies active queues. Rerun discovery to pick up newly completed families. Missing 10/50 results remain explicit.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/production_models.yaml').exists())
sys.path.insert(0, str(REPO / 'scripts'))
from ancestral_inspection import inventory, select_cohort, summarize_depth, bootstrap_curves, REPRESENTATIONS
RESULT_ROOTS = [REPO / 'runs/local_benchmarks/full_native_ancestral_20261001', REPO / 'runs/local_benchmarks/missing_alphabet_analysis_20261001']
COMMON_FAMILIES = True  # Intersection across representations with completed outputs.
MAX_FAMILIES = 50       # Quick inspection; set None for the full available cohort.
DEPTH_BINS = 10
BOOTSTRAP_DRAWS = 1000
SEED = 42
MIN_FAMILIES_PER_BIN = 5
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
COLORS = dict(zip(REPRESENTATIONS, plt.get_cmap('tab10').colors))

## Definitions

Depth = root-to-node branch distance / maximum root-to-tip distance in the family's own rooted tree. **0 is rootward; 1 is tipward.** This is normalized substitution distance, not time or matched nodes across strategies. Rooting uses recorded MAD/outgroup methods; inserted synthetic roots have no posterior and are omitted.

Maximum posterior probability (higher is sharper); normalized entropy H/log2(K) (lower is sharper); fraction with maximum probability >=0.90; uniform-baseline adjusted maximum (K*pmax-1)/(K-1). Normalization does not eliminate alphabet/model effects.

Keep occupancy-passing columns (at least four valid tips, <=30% missing). Average node/sites within family/depth bins, then weight families equally. Shading is a 95% **family bootstrap** interval, not an independent-site interval. Bins may have different contributing families; those below five are hidden by default. No cross-strategy node or column pairing.

In [ ]:
cohort = json.loads((REPO / 'configs/oma_benchmark_cohort.json').read_text())
family_ids = {row['family'] for row in cohort['families']}
available, problems = inventory(RESULT_ROOTS, family_ids)
coverage = available.groupby('representation').family.nunique().reindex(REPRESENTATIONS, fill_value=0)
display(coverage.rename('completed_families').to_frame().assign(expected_families=len(family_ids)))
if not problems.empty:
    display(problems)
    raise ValueError('Resolve invalid/conflicting artifacts before interpretation.')
selected = select_cohort(available, common=COMMON_FAMILIES, max_families=MAX_FAMILIES)
display(selected.groupby('representation').agg(families=('family', 'nunique'), nodes=('n_nodes', 'sum')))
print('Rooting:', selected.rooting.value_counts().to_dict())
print('Live incomplete sweep / deterministic subset: these are not final all-family results.')
print('Completion/provenance metadata checked; full output SHA256s are not recomputed during inspection.')

## Depth curves

Read gzip tables in chunks, retaining only family/bin summaries. The full 499-family analysis may take several minutes. Artifact pointers include inherited pilot results, avoiding duplicates. Change bins/family limit and rerun this section to refresh.

In [ ]:
family_bins, eligibility = summarize_depth(selected, bins=DEPTH_BINS)
curves = bootstrap_curves(family_bins, draws=BOOTSTRAP_DRAWS, seed=SEED)
display(eligibility.groupby('representation')[['total_node_sites', 'eligible_node_sites']].sum())
display(curves.query("tip_entropy_group == 'all' and metric == 'max_probability'").pivot(index='depth_midpoint', columns='representation', values='n_families').fillna(0).astype(int))

In [ ]:
def plot_curve(ax, metric, entropy_group='all'):
    table = curves[(curves.metric == metric) & (curves.tip_entropy_group == entropy_group)]
    for name, group in table.groupby('representation'):
        group = group.sort_values('depth_bin')
        supported = group.n_families >= MIN_FAMILIES_PER_BIN
        x = group.depth_midpoint.to_numpy()
        ax.plot(x, group['mean'].where(supported).to_numpy(), marker='o', markersize=3, color=COLORS[name], label=name)
        ax.fill_between(x, group.lower.where(supported).to_numpy(), group.upper.where(supported).to_numpy(), color=COLORS[name], alpha=.15)
    ax.set(xlabel='Normalized root-to-node distance (rootward → tipward)', ylim=(0, 1), xlim=(0, 1))
    ax.grid(alpha=.2)

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for ax, metric, title in zip(axes.flat, ['max_probability', 'normalized_entropy', 'sharpness', 'confidence_090'], ['Maximum posterior probability ↑', 'Normalized posterior entropy ↓', 'Uniform-baseline adjusted sharpness ↑', 'Fraction with maximum ≥0.90 ↑']):
    plot_curve(ax, metric)
    ax.set_title(title)
axes[0, 0].legend(fontsize=9)
plt.show()

## Conditioning on tip/column entropy

Split each strategy's own columns by normalized tip entropy: low <=1/3, medium <=2/3, high >2/3. These are strata, not paired homologous columns. Family coverage can be lower in each stratum.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, constrained_layout=True)
for ax, level in zip(axes, ['low', 'medium', 'high']):
    plot_curve(ax, 'max_probability', level)
    ax.set_title(f'{level.capitalize()} normalized tip entropy')
axes[0].set_ylabel('Maximum posterior probability')
axes[-1].legend(fontsize=8)
plt.show()

## Inspect one family and ancestral node

Edit FAMILY, REPRESENTATION and NODE. The scatter averages eligible sites within each internal node. The heatmap shows twelve most uncertain eligible sites of one node. Site numbers are **zero-based**, in this strategy's alignment only. Native FT2 labels are characters, not decoded amino acids.

In [ ]:
FAMILY = sorted(selected.family.unique())[0]
REPRESENTATION = 'FT2_20' if 'FT2_20' in set(selected.representation) else selected.representation.iloc[0]
entry = selected[(selected.family == FAMILY) & (selected.representation == REPRESENTATION)].iloc[0]
detail = pd.read_csv(entry.table)  # Full probability table for only one family.
detail = detail[detail.passes_occupancy == 1].copy()
if detail.empty:
    raise ValueError('No occupancy-passing sites in this family.')
nodes = detail.groupby('node').agg(depth=('normalized_root_depth', 'first'), max_probability=('max_probability', 'mean'), normalized_entropy=('normalized_entropy', 'mean'), descendant_tips=('descendant_tip_count', 'first'))
display(nodes.sort_values('depth').head(12))
fig, ax = plt.subplots(figsize=(7, 4))
points = ax.scatter(nodes.depth, nodes.max_probability, c=nodes.descendant_tips, cmap='viridis', s=45)
ax.set(xlabel='Normalized root-to-node distance', ylabel='Mean maximum posterior probability', title=f'{FAMILY} / {REPRESENTATION}', ylim=(0, 1))
fig.colorbar(points, ax=ax, label='Descendant tips')
plt.show()

In [ ]:
NODE = nodes.sort_values('depth').index[0]  # Replace with a node from the table.
sites = detail[detail.node == NODE].sort_values('normalized_entropy', ascending=False).head(12)
state_order = json.loads((Path(entry.directory) / 'summary.json').read_text())['state_order']
probability_columns = ['p_' + state for state in state_order]
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
im = ax.imshow(sites[probability_columns].to_numpy().T, aspect='auto', vmin=0, vmax=1, cmap='magma')
ax.set_xticks(range(len(sites)), labels=sites.site.astype(str))
ax.set_yticks(range(len(state_order)), labels=state_order)
ax.set(xlabel='Alignment site (zero-based)', ylabel='Native character', title=f'{FAMILY} / {REPRESENTATION} / {NODE}: most uncertain eligible sites')
fig.colorbar(im, ax=ax, label='Marginal posterior probability')
plt.show()
display(sites[['site', 'normalized_root_depth', 'max_probability', 'normalized_entropy', 'normalized_tip_entropy']])

## Interpretation checklist

Check bin/family coverage before ranking models. Sharper distributions can reflect alphabet size, fitted rates, rooting, composition or alignment differences, not improved ancestral accuracy. Repeat with MAX_FAMILIES=None when baselines finish. Available representations only define the current common cohort; FT2-10/50 appear when their native ASR jobs finish. Rooted normalized depth is not evolutionary time. Native ASR remains separate from concatenated fixed-species-tree controlled gain.